# LSTM Inference Walkthrough

Training and inference are different jobs. Training learns model parameters. Inference loads already-learned parameters and produces a forecast without changing the model.

This notebook walks through the saved LSTM inference path.

## What inference does not use

During inference there are no target labels, no loss, no gradients, no backpropagation, no optimizer steps, no epochs, and no early stopping. The model is put into `eval()` mode and predictions run under `torch.no_grad()`.

In [ ]:
# ruff: noqa: E402, I001
import json
import sys
from pathlib import Path

import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from market_resonance.inference import count_parameters, run_lstm_inference
from market_resonance.inference.lstm_inference import load_model_from_checkpoint

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "treasury_yields_daily.csv"
CHECKPOINT_PATH = PROJECT_ROOT / "reports" / "models" / "first_lstm.pt"
METRICS_PATH = PROJECT_ROOT / "results" / "inference_metrics.json"

## Load the saved checkpoint

The checkpoint contains learned model weights plus the metadata needed for inference, including feature columns, lookback length, horizon, and training normalization statistics.

In [ ]:
model, metadata = load_model_from_checkpoint(CHECKPOINT_PATH)
{
    "parameter_count": count_parameters(model),
    "lookback": metadata["lookback"],
    "horizon": metadata["horizon"],
    "num_features": len(metadata["feature_columns"]),
    "num_targets": len(metadata["target_columns"]),
    "model_training_mode": model.training,
}

`model.training` should be `False`, which means `model.eval()` has been applied.

## Run inference

The inference command rebuilds the latest 60-day feature sequence, applies the saved training normalization statistics, runs the model with no gradients, and saves forecast diagnostics.

In [ ]:
result = run_lstm_inference(
    data_path=DATA_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    metrics_path=METRICS_PATH,
)
{
    "latest_input_date": result.latest_input_date,
    "forecast_date": result.forecast_date,
    "parameter_count": result.parameter_count,
    "single_sample_latency_ms": result.single_sample_latency_ms,
    "batch_latency_ms": result.batch_latency_ms,
}

## Seven-maturity forecast

The model predicts future yield changes in percentage points. The inference layer converts those changes into basis points and adds them to the latest observed yields.

In [ ]:
forecast = pd.DataFrame(result.forecast)
forecast

## Metrics JSON

The same forecast and runtime diagnostics are saved for reproducibility.

In [ ]:
metrics = json.loads(METRICS_PATH.read_text())
metrics

## Why `torch.no_grad()` matters

In training, PyTorch tracks operations so it can compute gradients. In inference, we do not need gradients because no parameters are updated. `torch.no_grad()` makes prediction faster and uses less memory.

In [ ]:
with torch.no_grad():
    no_grad_is_enabled = not torch.is_grad_enabled()

no_grad_is_enabled